In [ ]:
import os
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from cbir.dataset import CbirDataset
from cbir.models import DatabaseFeatureExtractor

In [ ]:
BATCH_SIZE = 60
OXFORD_ROOT = "/home/ubuntu/data/datasets/roxford5k/jpg"
CACHE_DIR = "/home/ubuntu/data/feature_cache"

In [ ]:
oxford_dataset = CbirDataset(OXFORD_ROOT)
oxford_dataloader = DataLoader(oxford_dataset, batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
db_extractor = DatabaseFeatureExtractor(num_features=500, num_clusters=10).to(device)
db_extractor.eval()

with torch.no_grad():
    print(f"Processing roxford5k....")
    oxford_feature_cache = {}

    for batch_images, batch_ids in tqdm(oxford_dataloader):
        batch_images = batch_images.to(device)
        
        batched_clusters, batched_centroids, batched_radii = db_extractor(batch_images)
        
        batched_clusters = batched_clusters.cpu()
        batched_centroids = batched_centroids.cpu()
        batched_radii = batched_radii.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]

            oxford_feature_cache[image_id] = {
                'clusters': batched_clusters[i],
                'centroid': batched_centroids[i],
                'radius': batched_radii[i]
            }

In [ ]:
torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "roxford5k_features_pruned.pkl"))